# IOI. Activation patching

Wang et al., *Interpretability in the Wild: a Circuit for Indirect Object Identification*, ICLR 2023 — [arXiv:2211.00593](https://arxiv.org/abs/2211.00593). Hands-on рядом: ARENA, раздел IOI; Neel Nanda, *Exploratory Analysis Demo*.

Узел дерева: Interpretability → causal → activation patching (`i_actpatch`).
Модель: GPT-2 small, CPU. ~15 forward-проходов, секунды; разовая загрузка весов ~500 МБ, ~2 ГБ RAM.

Цель: понять activation patching, реализовав его на узком срезе — локализация по слоям, где формируется ответ. Полный head/path-patching и схема из 26 голов — узел `i_pathpatch`.

План: Part 1 — метрика logit-diff; Part 2 — patching; Part 3 — свип по слоям. Застрянешь — внизу есть **SPOILERS** с пошаговым рецептом.

## Эксперименты-первыми

Открой в статье разделы faithfulness / completeness / minimality и заполни до чтения метода.

- Что показали результаты против того, что обещает abstract?
- Чем бы фальсифицировала схему из 26 голов (minimality)?
- Что не проверено: одна задача, одна модель — что с переносом?

## Зачем это

**Прикладная значимость.** Поведенческие метрики говорят, что модель выдала, но не почему. Чтобы править модель без переобучения (model editing), убирать нежелательное поведение (bias, обман), строить eval-ы по механизму, а не по поверхности, и аудировать систему — нужно знать внутренние причинно ответственные компоненты. Activation patching их находит.

**Почему такой метод.** Линейная проба показывает, что признак линейно декодируется из активаций. Это корреляция: признак может присутствовать и не использоваться. Patching вмешивается в вычисление и меряет, изменился ли ответ — только так получаешь право сказать, что компонент причинно отвечает за поведение.

**Почему такой toy-сеттинг.** IOI — чистый контролируемый полигон: правильный и сломанный ответ заданы явно (меняем John и Mary), схема мала настолько, что разбирается целиком и проверяется (faithfulness, minimality). На нём метод отрабатывают и валидируют, прежде чем нести на грязные реальные задачи. Цель исследования — показать, что поведение модели раскладывается на конкретную, проверяемую причинную схему компонентов, а не остаётся чёрным ящиком. IOI здесь тренажёр техники, не цель.

## Теория

**Задача.** «When John and Mary went to the store, John gave a drink to ___». Ответ — Mary (indirect object). Сложность: John встречается дважды (субъект), и модель должна подавить повторённое имя и скопировать одиночное.

**Residual stream.** Каждый блок (attention, MLP) читает из остаточного потока и дописывает свой вклад; финальные логиты считаются как $\text{logits} = W_U \cdot \text{LN}(\text{resid\_final})$. Вклады складываются линейно — это даёт приписывать эффект конкретным местам.

**Activation patching.** Два прогона: clean (ответ Mary) и corrupt (поменяли John↔Mary, ответ ломается). Подменяем активацию в одной точке значением из другого прогона и смотрим сдвиг метрики. Мы вмешиваемся в вычисление, поэтому это причинный тест; проба по hidden state только читает активацию и даёт корреляцию.

**Метрика — logit difference.** $\text{LD} = \text{logit}(\text{Mary}) - \text{logit}(\text{John})$ на последнем токене. Линейна по вкладам и прямо отражает поведение; softmax насыщается и хуже раскладывается. Эквивалентно это проекция $\text{resid\_final}$ на направление $W_U[\text{Mary}] - W_U[\text{John}]$.

**Направление (denoising).** Бежим corrupt, патчим clean-активацию, меряем $\text{restored} = \text{patched} - \text{corrupt\_base}$. Полное восстановление равно clean-уровню `base`.

**Что локализуем.** Патч `resid_post[layer]` на позиции −1 показывает, на какой глубине ответ уже дописан: до нужного слоя восстановление частичное, после — полное. Слой (где) и голова (кто) — разные вопросы; головы это `i_pathpatch`.

**Полная схема (контекст).** Анализ статьи раскладывает поведение на 26 голов в 7 классов: duplicate token и induction heads ловят повтор имени, S-inhibition heads гасят субъект, name mover heads (слои ~9–10) копируют оставшееся имя в логиты. Здесь воспроизводим только грубую локализацию по слоям.

*Где у меня дыра в понимании:* …

❓ Inline Question 1. Чем activation patching причинно сильнее линейной пробы по hidden state?

In [ ]:
# setup (запусти как есть; первый раз качает веса ~500 МБ)
# pip install transformer_lens matplotlib
import torch
from transformer_lens import HookedTransformer, utils
torch.set_grad_enabled(False)

model = HookedTransformer.from_pretrained("gpt2", device="cpu")  # 12 слоёв, d_model=768
clean   = "When John and Mary went to the store, John gave a drink to"
corrupt = "When John and Mary went to the store, Mary gave a drink to"
clean_tokens   = model.to_tokens(clean)    # shape: [1, seq]
corrupt_tokens = model.to_tokens(corrupt)  # shape: [1, seq]
MARY = model.to_single_token(" Mary")
JOHN = model.to_single_token(" John")
print(model.to_str_tokens(clean)[-4:], "ids Mary,John:", MARY, JOHN)
# если что-то идёт не так дальше — дебагай принтами: распечатывай shapes и значения

## Part 1. Метрика logit-diff

Метрика, которой меряем всё дальше: насколько на последнем токене модель предпочитает Mary над John. Берём последний токен — там предсказывается следующее слово (ответ).

In [ ]:
def logit_diff(logits):
    # YOUR CODE: верни скаляр logit(' Mary') - logit(' John') на последнем токене.
    # logits shape: [batch, seq, d_vocab]; нужен logits[0, -1, :], индексы MARY и JOHN.
    raise NotImplementedError

base = logit_diff(model(clean_tokens))
print("base logit_diff =", float(base))

In [ ]:
b = float(base)
print(f"Expected ≈ 3.3 (норма 2–5) | у тебя {b:.3f}")
assert 2.0 < b < 5.0, "вне нормы: скорее всего берёшь не последний токен (индекс -1) или перепутал MARY/JOHN"
print("ok")

❓ Inline Question 2. Почему разница логитов, а не softmax-вероятность Mary? Что это даёт при суммировании вкладов компонентов (логиты аддитивны по residual stream)?

## Part 2. Activation patching

Одна интервенция:

```
clean:   resid_post[L] (pos -1) = c   ->  ln+unembed  ->  logit_diff = base
corrupt: resid_post[L] (pos -1) = k   ->  ln+unembed  ->  logit_diff = corrupt_base
patch:   бежим corrupt, но в точке (L, -1) подставляем c из clean  ->  logit_diff = ?
```

`resid_post[L]` — остаточный поток сразу после блока `L`. Внутри corrupt-прогона подменяем его значение на последней позиции значением из clean-кэша. Если точка причинно несёт ответ, метрика подтянется к `base`.

In [ ]:
_, clean_cache = model.run_with_cache(clean_tokens)   # clean_cache[name]: [batch, seq, d_model]
corrupt_base = logit_diff(model(corrupt_tokens))
print("corrupt base =", float(corrupt_base))

def patched_logit_diff(layer):
    # YOUR CODE: logit_diff corrupt-прогона с пропатченным resid_post[layer] на позиции -1.
    #   1) name = utils.get_act_name('resid_post', layer)
    #   2) def hook(act, hook): act[:, -1, :] = clean_cache[hook.name][:, -1, :]; return act
    #      (act shape: [batch, seq, d_model] -> пишем только позицию -1)
    #   3) logits = model.run_with_hooks(corrupt_tokens, fwd_hooks=[(name, hook)])
    #   4) return logit_diff(logits)
    raise NotImplementedError

In [ ]:
# Тождество: после resid_post последнего слоя на позиции -1 не считается ничего,
# кроме ln_final и unembed этой позиции. Значит патч clean сюда даёт ровно clean.
L = model.cfg.n_layers - 1
restored = float(patched_logit_diff(L))
rel = abs(restored - float(base)) / (abs(float(base)) + 1e-9)
print(f"Expected \u2248 {float(base):.3f} | у тебя {restored:.3f} | rel_err {rel:.2e}")
assert rel < 1e-3, "патч последнего слоя обязан дать ~clean: скорее всего пишешь не в позицию -1, либо берёшь не из clean_cache"
print("ok")

❓ Inline Question 3. Почему патч последнего слоя восстанавливает logit-diff полностью, а ранних — частично? Где дописывается ответ?

## Part 3. Свип по слоям

Прогоняем `patched_logit_diff` по всем слоям. Форма кривой говорит, на какой глубине ответ становится решённым.

In [ ]:
import matplotlib.pyplot as plt
eff = [float(patched_logit_diff(l)) - float(corrupt_base) for l in range(model.cfg.n_layers)]
plt.figure(figsize=(7, 3))
plt.plot(range(model.cfg.n_layers), eff, marker="o")
plt.xlabel("layer"); plt.ylabel("restored (patched - corrupt)")
plt.title("Activation patching по слоям, resid_post, last pos"); plt.grid(True); plt.show()
# resid_post кумулятивен -> кривая растёт монотонно; интересен не argmax уровня (это последний слой),
# а слой с НАИБОЛЬШИМ скачком восстановления (там доформировывается ответ).
jumps = [eff[l] - eff[l-1] for l in range(1, model.cfg.n_layers)]
best = 1 + max(range(len(jumps)), key=lambda i: jumps[i])
print("наибольший скачок восстановления на слое", best)

Name mover heads в GPT-2 small сидят в слоях ~9–10; там ожидается наибольший скачок восстановления. Где скачок у тебя, совпал ли?

Caveat: мы корраптим position-swap (John↔Mary), а не каноническим ABC-корраптом из статьи (третье имя). Это валидный proxy, но совпадение со слоями/головами статьи будет приблизительным именно из-за выбора контрфактуала.

**[bonus]** Поменяй пару имён на (' Alice', ' Bob') и прогони свип заново. Форма держится? Если да — схема про роль (subject/IO), а не про конкретные токены.

## SPOILERS — рецепт, если застряла

Сначала попробуй сама. Если затык:

- **logit_diff:** `lg = logits[0, -1]; return (lg[MARY] - lg[JOHN]).item()`.
- **hook:** активация приходит формой `[batch, seq, d_model]`; меняешь только срез `[:, -1, :]` и возвращаешь тензор. Имя точки берёшь из `hook.name`, оно же ключ в `clean_cache`.
- **NaN / странные числа:** проверь, что `torch.set_grad_enabled(False)` отработал и что не перепутал `clean`/`corrupt` прогоны. Прогони один слой и распечатай `restored`.

## Рефлексия

- Что переносимо в твою работу.
- Конкретный перенос: что взять за clean и corrupt, какую активацию или вход патчить, чтобы интервенция меняла причину, а не следствие (не полу-циркулярно), какой результат подтвердил бы причинность.
- Что осталось неясным → следующий узел: `i_pathpatch` (от слоёв к головам).